# 01 — Business Understanding
### AI Clinical Decision Support System for Alzheimer's Disease Prediction

**Author:** Data Science Project Team
**Dataset:** P685 Alzheimer's Disease Data (2,149 de-identified patient records)
**Notebook purpose:** Frame the business/clinical problem, objectives, success criteria, and constraints before touching the data, following a CRISP-DM style workflow.


## 1.1 Background

Alzheimer's disease (AD) is a progressive neurodegenerative condition and the most common cause of dementia. Early identification of patients at elevated risk allows clinicians to prioritise further cognitive assessment, initiate care planning earlier, and potentially enrol patients in interventions while they are most effective.

In practice, clinicians already collect a wide range of demographic, lifestyle, clinical, and cognitive-screening data during routine visits (e.g. MMSE score, blood pressure, cholesterol panel, family history, memory complaints). The volume and complexity of this information makes it hard to consistently weigh every risk factor for every patient, especially in high-volume primary care settings.

This project builds a **clinical decision support system (CDSS)**: a machine learning model that ingests routinely-collected patient data and outputs a probability-based "Alzheimer's risk flag", along with an explanation of *why* the model produced that flag, to support - not replace - clinical judgement.

## 1.2 Business / Clinical Objectives

1. **Primary objective:** Build a classification model that predicts the binary `Diagnosis` outcome (Alzheimer's / No Alzheimer's) from patient demographic, lifestyle, clinical, and cognitive data, with strong discriminative performance (target: ROC-AUC ≥ 0.85).
2. **Secondary objective:** Prioritise **recall** (sensitivity) for the positive class in the model selection process. In a screening context, a false negative (missing a true Alzheimer's case) is clinically more costly than a false positive (an unnecessary follow-up cognitive assessment).
3. **Explainability objective:** Every prediction must be explainable at both the population level (which features drive risk generally) and the individual patient level (why was *this* patient flagged), so a clinician can sanity-check and trust the output.
4. **Deployment objective:** Package the final model behind a simple interactive application (Streamlit) so a non-technical clinical user can enter a patient's data and receive a risk flag + explanation without touching code.

## 1.3 Success Criteria

| Criterion | Target |
|---|---|
| Test-set ROC-AUC | ≥ 0.85 |
| Test-set Recall (Alzheimer's class) | ≥ 0.80 |
| Explainability | Global + local (per-patient) SHAP explanations available |
| Reproducibility | Fixed random seeds, versioned processed data, saved model artefact |
| Usability | Working Streamlit app taking raw clinical inputs |

## 1.4 Constraints & Assumptions

- The dataset is a **single, de-identified, cross-sectional extract** (2,149 patients) - not a longitudinal or multi-site dataset. Findings and model performance should be understood as *internally* valid only; **external clinical validation is required before any real deployment**.
- `MMSE`, `FunctionalAssessment`, and `ADL` are themselves clinician-administered assessment scores. The model is therefore a **decision-support layer on top of existing clinical workflow**, not an independent diagnostic replacement.
- `DoctorInCharge` is a placeholder/confidentiality column (`XXXConfid`) in this extract and carries no signal - it will be dropped.
- This is an educational / portfolio-grade project (not a validated medical device). All outputs should be framed as **decision support**, never as a standalone diagnosis.

## 1.5 Project Plan (CRISP-DM aligned)

| Phase | Notebook |
|---|---|
| Business Understanding | `01_Business_Understanding.ipynb` (this notebook) |
| Data Understanding | `02_Data_Understanding.ipynb` |
| Data Quality Profiling | `03_Data_Profiling.ipynb` |
| Exploratory Data Analysis | `04_Exploratory_Data_Analysis.ipynb` |
| Feature Engineering | `05_Feature_Engineering.ipynb` |
| Model Development | `06_Model_Development.ipynb` |
| Evaluation & Explainability | `07_Model_Evaluation_Explainability.ipynb` |

Supporting reusable logic lives in `src/` (data validation, preprocessing, feature engineering, models, evaluation, explainability, visualization, utils) so every notebook calls tested functions rather than duplicating code. The final trained model is saved to `models/`, and a working demo app lives in `app/`.

In [1]:
import sys
sys.path.insert(0, '..')
from src.utils import RAW_DATA_PATH, TARGET_COL
import pandas as pd

df_preview = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset shape: {df_preview.shape}")
print(f"Target column: '{TARGET_COL}'")
df_preview.head()

Dataset shape: (2149, 35)
Target column: 'Diagnosis'


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


## 1.6 Data Dictionary (high level)

| Group | Columns |
|---|---|
| Identifiers (dropped before modelling) | `PatientID`, `DoctorInCharge` |
| Demographics | `Age`, `Gender`, `Ethnicity`, `EducationLevel` |
| Lifestyle | `BMI`, `Smoking`, `AlcoholConsumption`, `PhysicalActivity`, `DietQuality`, `SleepQuality` |
| Medical history | `FamilyHistoryAlzheimers`, `CardiovascularDisease`, `Diabetes`, `Depression`, `HeadInjury`, `Hypertension` |
| Clinical measurements | `SystolicBP`, `DiastolicBP`, `CholesterolTotal`, `CholesterolLDL`, `CholesterolHDL`, `CholesterolTriglycerides` |
| Cognitive & functional assessment | `MMSE`, `FunctionalAssessment`, `ADL` |
| Symptoms | `MemoryComplaints`, `BehavioralProblems`, `Confusion`, `Disorientation`, `PersonalityChanges`, `DifficultyCompletingTasks`, `Forgetfulness` |
| **Target** | `Diagnosis` (0 = No Alzheimer's, 1 = Alzheimer's) |

Full statistical profiling of each of these groups happens in `02_Data_Understanding.ipynb` and `03_Data_Profiling.ipynb`.